# పాఠం 05 - ఏజెంటిక్ RAG


## సెటప్

ఈ నోట్‌బుక్ మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఉపయోగించి ఏజెంటిక్ RAG (రిట్రీవల్-ఆగుమెంటెడ్ జనరేషన్) నమూనాను చూపిస్తుంది.

**ఆవశ్యకతలు:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — మీ Azure AI సెర్చ్ సర్వీస్ ఎండ్పాయింట్
- `AZURE_SEARCH_API_KEY` — మీ Azure AI సెర్చ్ API కీ
- వాతావరణ చరాలను ఉపయోగించి అమర్చబడిన Azure OpenAI డిప్లాయ్‌మెంట్
- Azure CLI సత్యాపితమైనది (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ఏజెంటిక్ RAG అంటే ఏమిటి?

సాంప్రదాయ RAG ఒక స్థిరమైన పైప్‌లైన్‌ను అనుసరిస్తుంది: డాక్యుమెంట్లను తీసుకుని, ఆపై స్పందనను ఉత్పత్తి చేయడం. **ఏజెంటిక్ RAG** ఆజెంట్ కి సమాచారం తీసుకునే **ఎప్పుడు** మరియు **ఎలా** నిర్ణయించుకునే స్వేచ్ఛని ఇస్తుంది.

ఏజెంటిక్ RAG తో, ఆజెంట్ చేయగలదు:
- ప్రశ్నకు సమాధానం ఇవ్వక ముందు **తీసుకునే అవసరం ఉందా** అని **నిర్ణయించు**
- ఏ డేటా మూలం లేదా టూల్ ను ప్రశ్నించాలో **ఎంచుకోండి**
- తీసుకున్న ఫలితాలను **మూల్యాంకనం చేయండి** మరియు మొదటి ప్రయత్నం తగినంత కాకుంటే అనుసరణ retrievals చేయండి
- అనేక retrieval దశల నుండి సమాచారాన్ని సమగ్రమైన సమాధానంలో **సంధించు**

ఇది ఆజెంట్ ను స్థిరమైన retrieve-తర్వాత-ఉత్పత్తి చేయు పైప్‌లైన్ కంటే అంతకుమించిన మృదువుగా మరియు ఖచ్చితంగా చేస్తుంది.


## శోధన టూల్ సృష్టించడం

Agentic RAGలో, బాహ్య డేటా వనరులను ఏజెంట్ డిమాండ్ పై పిలవగల **టూల్స్**గా మలచబడతాయి. ఇది ఏజెంట్‌కి రిట్రీవల్‌ని ఒక ఆప్షనల్ చర్యగా తీసుకోవడానికి, తప్పనిసరి దశగా కాకుండా చేయిస్తుంది.

దిగువన మేము ఒక ప్రయాణ జ్ఞాన ఆధారాన్ని నిర్వచించి, దాన్ని ఏజెంట్ తిరిగి డెస్టినేషన్ సమాచారాన్ని చూడటానికి పిలవగల టూల్‌గా అందిస్తాము.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ఏజెంట్‌ను తయారు చేయడం

ఇప్పుడు మేము ఎల్లప్పుడూ సమాధానం ఇచ్చే ముందు సమాచారాన్ని **ఎల్లప్పుడూ పొందమని** ఆదేశించబడిన ఏజెంట్‌ను సృష్టిస్తాము. ఏజెంట్ తన స్వంత శిక్షణ డేటాకు ఆధారపడకుండా జ్ఞాన ఆధారంగా తన స్పందనలను స్థిరపరిచేందుకు `search_travel_knowledge` టూల్‌ను ఉపయోగిస్తుంది.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## తిరిగి పొందడం — తయారుచేసే వ్యక్తి-పరీక్షించే వ్యక్తి నమూనా

ఏజెంటిక్ RAG యొక్క ఒక ప్రధాన లాభం **పునరావృత తిరిగి పొందడం**. ఏజెంట్ తన ప్రారంభ కనుగొనಲುలను నిర్ధారించడానికి, మెరుగుపరచడానికి లేదా విస్తరించడానికి అనేక సార్లు శోధన చేయగలదు — ఇది "తయారుచేసే వ్యక్తి-పరీక్షించే వ్యక్తి" పని విధానానికి సమానం:

1. **తయారుచేసే వ్యక్తి దశ**: ఏజెంట్ ప్రారంభ సమాచారాన్ని తెచ్చి, ఒక ప్రతిస్పందనను రూపకల్పన చేస్తుంది.
2. **పరీక్షించే వ్యక్తి దశ**: ఏజెంట్ వివరాలను నిర్ధారించడానికి లేదా లోట్లను పోగొట్టడానికి అదనపు తిరిగి పొందడాలను చేస్తుంది.

క్రింద, ఏజెంట్‌ను ఒక ప్రశ్న అడుగుతారు, ఇది అనేక ప్రదేశాలను పోల్చడాన్ని అవసరం చేస్తుంది, కాబట్టి అది అనేక సార్లు శోధించడానికి ప్రేరేపిస్తుంది.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ను ఉపయోగించి **Agentic RAG** వ్యవస్థను తయారు చేయటం ఎలా అనేది నేర్చుకున్నారు:

- **Agentic RAG** ఏజెంట్లు స్వయంచాలకంగా ఎప్పుడు సమాచారం శోధించాలని నిర్ణయించుకునేలా చేస్తుంది, ఇది శోధనను స్థిరంగా కాకుండా చలనం గామ్యంగా చేస్తుంది.
- **సాధనాలను డేటా మూలాలుగా ఉపయోగించడం**: బాహ్య పరిజ్ఞాన ఆధారాలు (అట్లాంటిది Azure AI Search) ను ఏజెంట్లు ఉపయోగించగల సాధనాలుగా మడతబెట్టి ఉంచుతారు.
- **పునరావృతశోధన**: తయారీదారు-తగ్గింపు నమూనా ఏజెంట్కు పలు శోధన రౌండ్లు — శోధన, ధృవీకరణ, మరియు శ్రద్ధపరచటం — చేయటానికి అనుమతిస్తుంది అంటే తుది సమాధానం ఇస్తుంది.

ఉత్పత్తిలో, మీరు మెమరీలో ఉన్న `TRAVEL_KNOWLEDGE_BASE` ని పెద్ద స్థాయి ప్రయాణ పత్రాల శోధనను నిర్వహించేందుకు నిజమైన Azure AI Search ఇండెక్స్ తో మార్చాలి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
